# Mission 2: The Missing Clue

Compare two prepared scenes, then run a pretrained YOLO ONNX model on the real-photo samples. A model score is confidence, not certainty. COCO supports common classes such as people, backpacks, cups and cell phones, but it does not know our fictional prototype.

## Learning objectives and setup
Learn inference, labels, confidence, boxes, counts, crops and sampled video frames. From the repository root, install core packages with `python -m pip install -e ".[test]"`. For actual YOLO inference, run `python -m pip install -e ".[model]"` and `python scripts/setup_model.py`. Inputs include the staged scenes and three CC0 photo samples. Scene comparison remains an honest fallback if weights are unavailable.

Exercise: predict what changing confidence from 0.35 to 0.7 could remove.

In [ ]:
from pathlib import Path
import sys, cv2
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from the_last_chai.image_enhancement import load_image
from the_last_chai.object_detection import load_model, infer_image, infer_video_frames, annotate_video, count_objects, draw_detections, crop_detection
# Each function operates on the actual image/video and returns detections, not a fixed answer.
early = load_image(str(ROOT / 'assets/images/cctv_corrupted.png'))
later = load_image(str(ROOT / 'assets/images/lab_reference_2343.png'))
plt.figure(figsize=(12,4)); plt.subplot(1,2,1); plt.imshow(cv2.cvtColor(early, cv2.COLOR_BGR2RGB)); plt.title('23:42 staged still'); plt.axis('off'); plt.subplot(1,2,2); plt.imshow(cv2.cvtColor(later, cv2.COLOR_BGR2RGB)); plt.title('23:43 reference scene'); plt.axis('off');

A pretrained detector converts an image to a tensor, runs a neural network, then filters overlapping boxes with library-provided Non-Maximum Suppression. `infer_image` returns only classes actually emitted by the model. Missing weights raise an actionable error rather than inventing results. If you have a standard 640×640 YOLOv8 COCO ONNX export, place it at `models/yolov8n.onnx`; decoder support covers the common 84-by-candidates YOLOv8 output and the 85-column YOLOv5 output. Validate custom exports before relying on results.

In [ ]:
weights = ROOT / 'models/yolov8n.onnx'
try:
    net = load_model(str(weights))
    detections = infer_image(net, later, confidence=0.35)
    print('Actual model detections:', detections)
    print('Counts from detections:', count_objects(detections))
    annotated = draw_detections(later, detections)
    plt.figure(figsize=(10,5)); plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)); plt.axis('off');
except FileNotFoundError as error:
    net = None
    print('Model unavailable:', error)
    print('Fallback only: compare the two staged scene images and the ground-truth manifest. These are not model detections.')

In [ ]:
if net is not None:
    photo_samples = {
        'person with backpack': ROOT / 'assets/images/real_red_backpack_person.jpg',
        'person using smartphone': ROOT / 'assets/images/real_smartphone_use.jpg',
        'coffee cup': ROOT / 'assets/images/real_coffee_cup.jpg',
    }
    for label, path in photo_samples.items():
        sample = load_image(str(path))
        predictions = infer_image(net, sample, confidence=0.25)
        print(label, 'actual predictions:', count_objects(predictions), predictions)
        preview = draw_detections(sample, predictions)
        plt.figure(figsize=(7, 4)); plt.imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB)); plt.title(label); plt.axis('off');
else:
    print('Install the optional model extra and run scripts/setup_model.py for live inference.')

In [ ]:
# TODO: Change confidence to 0.6, then 0.25. Compare actual labels and counts.
if net is not None:
    threshold_trial = infer_image(net, load_image(str(ROOT / 'assets/images/real_smartphone_use.jpg')), confidence=0.6)
    print('Your threshold result:', threshold_trial)
    print('Counts:', count_objects(threshold_trial))
else:
    print('Prepare the optional model to run this inference exercise.')

Bounding boxes crop pixel regions, not semantic truth. Video frame sampling runs the same image inference on every Nth frame and releases the video handle. A cup may not be detected; do not treat absence from detector output as proof an object is absent.

Exercise: if weights are present, sample every 10th frame and compare the labels with the scene inventory. Hint 1: inspect `models/`. Hint 2: use `infer_video_frames(net, video_path, every_n_frames=10)`. Hint 3: model input/output export format must match the decoder.

Clue: the prepared later scene adds a cup; the phone/prototype absence is a scene-inventory story fact. Challenge: why can a high confidence score still be wrong? Common errors: wrong ONNX output shape, missing weights, or low-resolution objects. No model output is claimed without actual weights and inference.

In [ ]:
if net is not None:
    video_path = ROOT / 'assets/videos/corridor_staged.mp4'
    sampled = infer_video_frames(net, str(video_path), every_n_frames=10)
    print('Sampled frames:', len(sampled), 'first result:', sampled[0] if sampled else None)
    if detections:
        crop = crop_detection(later, detections[0])
        plt.figure(figsize=(5, 4)); plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)); plt.axis('off');
    video_result = annotate_video(net, str(video_path), str(ROOT / 'outputs/level2_annotated_video.mp4'))
    print('Actual model-annotated video:', video_result)

## Recap
We compared staged scenes and provided a genuine optional inference route. The local repository does not include large model weights. COCO labels are limited and do not include the fictional prototype; the prepared inventory comparison is explicitly not AI inference.